# Workshop problems in Theoretical Neuroscience – Week 2

**Rune W. Berg¹, Rasmus Herlo²**

¹ Department of Neuroscience  
² Center for Translational Neuromedicine

Faculty of Health and Medical Sciences, University of Copenhagen, Blegdamsvej 3, 2200 Copenhagen, Denmark.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
import math
from collections.abc import Iterable
import itertools
from typing import Iterable

import networkx as nx
import numpy as np
import numpy.typing as npt

from neuro import plotting

# 1 Epistemology, Information Theory and Graph-Modelling

## 1.1 Information Theory

### 1.1.1 Pen-and-paper exercises

**Shannon entropy**

The Shannon entropy of a discrete random variable $X$ with outcomes $\{x_i\}$ and probabilities $\{p_i\}$ is defined as

$$
H(X) = -\sum_i p_i \log_2 p_i
\tag{1}
$$


***

1. A fair coin is flipped. What is $H$? Now compute $H$ for a biased coin with $p(\text{heads}) = 0.9$. Which case carries more information per flip, and why?

In [ ]:
def entropy(probs: Iterable[float]) -> float:
    pvals = list(probs)
    assert abs(sum(pvals) - 1) < 0.00001 and all(p >= 0 for p in probs)
    # This is just to allow p=0 as well
    contributions = (0 if p == 0 else -p*math.log2(p) for p in pvals)
    res = sum(contributions)
    return res


def entropy_bernoulli(p: float) -> float:
    return entropy([p, 1.0 - p])


def entropy_uniform(n: int) -> float:
    return entropy([1.0/n for _ in range(n)])


print("H for 50/50:", entropy([0.5, 0.5]))
print("H for 90/10:", entropy([0.9, 0.1]))

The fair one contains more information. The log term (self-information), $\log(p_i)$ increases for low values of $p_i$, but the factor of $p_i$ decreases faster

2. A neuron fires with probability $p$ at each time step. Compute and plot $H(p)$ as a function of $p \in [0,1]$. At what value of $p$ is entropy maximised?

In [ ]:
# NOTE: 'in each timestep' is a bit unclear. Assuming two

ax = plotting.plot_function(
    func=entropy_bernoulli,
    xmin=0,
    xmax=1
)
ax.axvline(x=0.5, color="black", linestyle="dotted");

Behold! It maxes out at 0.5

3. A neuron can be in one of four equally likely states. What is $H$? How does this compare to a two-state neuron?

In [ ]:
print(entropy_uniform(4))
print(entropy_uniform(2))

It seems the 4-state system has a bit more information heeehehehehsorry

***

**Information and uncertainty**

Consider Shannon’s definition: information is the reduction of uncertainty.

1. You observe a neuron that fires on every single trial without exception. How much information does each spike carry? Justify your answer using the entropy formula.

2. You receive a message from a system with 8 equally likely symbols. How many bits are required to encode one symbol? What if the system had 1024 equally likely symbols?

3. Briefly explain in your own words the difference between the syntactic (Shannon) view and the semantic (phenomenological) view of information. Give a concrete biological example where the syntactic view is sufficient, and one where it is not.

1. Zero, because the terms in H go to zero when p goes to zero or one
2. 3 bits for 8. 10 bits.for 1024
3. Syntactic: Just bits - amount of zeros and ones that are transmitted by a signal. Semantic: Closer to the concept of information in common language - depends on context, meaning, and intention. Example: Syntactic is great when assessing e.g. information flow through a small ensemble of neurons, but not for addressing e.g. questions of 'information to do X', which aren't reducible to 'enough bits to do X'.

### 1.1.2 Jupyter exercises

**Plotting entropy**

Implement and plot Shannon entropy

$$
H(p) = -p\log_2 p - (1-p)\log_2(1-p)
$$

for a two-outcome system as a function of $p \in (0,1)$. Mark the maximum. The plot was then extended to a system with $n=4$ equally probable outcomes for varying $p$ of one outcome (distributing the remaining probability equally among the others). How does the shape change?

In [ ]:
# NOTE: the plot 'was then extended'?
_, ax = plt.subplots()

n_outcomes = (2, 4, 8, 9, 32)

for n in n_outcomes:
    pvals = np.linspace(0.0, 1.0, num=1000)
    probs = ((p, *((1.0 - p)/(n-1) for _ in range(n-1))) for p in pvals)
    entropies = [entropy(args) for args in probs]
    H_max = max(entropies)

    color = plotting.pyplot_utils.cycle_color(ax)

    ax.plot(pvals, entropies, label=f"$n={n}$ ($H_{{max}} = {H_max:.3g}$)", color=color)
    ax.axhline(y=H_max, color=color, alpha=0.5, linestyle="dashed")


ax.legend()
ax.set_xlabel("$p_0$")
ax.set_ylabel("$H$");

So, still the same maximum entropy ($\log_2(n)$). Wrt the shape, the max is pushed towards the left as n increases. Makes sense bc as $p_0$ tends to zero, the entropy has to approach that of $n-1$ uniformly distributed outcomes (cf. the line for $n=9$, which approaches $\log_2(8) = 3$ bits)

***

**Simulating a noisy channel**

Simulate a binary symmetric channel: send 10,000 random bits, flip each bit independently with error probability $\varepsilon$. Plot the fraction of errors as a function of $\varepsilon \in [0,0.5]$. At what value of $\varepsilon$ does the channel become completely uninformative and why?

In [ ]:
def mutual_information(bits_sent: npt.NDArray[np.int_], bits_received: npt.NDArray[np.int_]) -> float:
    """Computes the mutual information from arrays of sent and received bits"""

    n_bits = len(bits_sent)
    assert len(bits_received) == n_bits

    # Determine H_x - entropy of sent bits
    sent_ones_mask = bits_sent == 1
    sent_ones_frac = sum(sent_ones_mask) / n_bits
    H_x = entropy_bernoulli(sent_ones_frac)

    # Determine conditional entropy of sent bits given received bits
    H_xy = 0.0
    received_ones_mask = bits_received == 1

    # Condition on received bits being 1 or 0
    for rmask in (received_ones_mask, ~received_ones_mask):
        # Get the sent bits which were received as this value
        match = bits_sent[rmask]
        # Compute the entropy among the sent bits received as the value
        frac_ones_sent = sum(match)/len(match)
        # Add contribution
        p_y = len(match) / n_bits
        H_xy += p_y*entropy_bernoulli(frac_ones_sent)

    return float(H_x - H_xy)
    

rs = np.random.RandomState(seed=42)
error_rates = np.linspace(0.0, 0.5, num=100)
error_fracs: list[float] = []
mi_vals: list[float] = []

for epsilon in error_rates:

    bits_sent = rs.choice([0, 1], size=10_000)

    noise = rs.choice([1, 0], size=len(bits_sent), p=[epsilon, 1.0-epsilon])
    bits_received = (bits_sent + noise) % 2

    error_fracs.append(noise.mean())

    mi_vals.append(mutual_information(bits_sent=bits_sent, bits_received=bits_received))


_, ax = plt.subplots()


line_mi, = ax.plot(error_rates, mi_vals, label="Mutual information (left axis)")
ax2 = ax.twinx()
line_err, = ax2.plot(
    error_rates,
    error_fracs,
    label="Observed error rate (right axis)",
    color="black",
    alpha=0.4
    )

ax.legend(handles=[line_mi, line_err])
ax.set_xlabel("$\\varepsilon$")
ax.set_ylabel("bits");

It becomes uninformative at $\varepsilon = 0.5$. Makes sense because that's when receiving e.g. a one is equally likely to correspond to a one or a zero being sent. 

***

## 1.2 The Neuron as an Information Processing Unit

### 1.2.1 Pen-and-paper exercises

**The McCulloch–Pitts neuron**

The McCulloch–Pitts (1943) neuron computes

$$
y = \Theta\left(\sum_i w_i x_i - \theta\right)
\tag{2}
$$

where $\Theta$ is the Heaviside step function, $w_i$ are weights, $x_i \in \{0,1\}$ are binary inputs, and $\theta$ is a threshold.


In [ ]:
class Perceptron:
    """Simulated McCulloch–Pitts Neuron."""

    def __init__(self, weight: Iterable[float], threshold: float) -> None:
        self.weight = np.fromiter(weight, dtype=float)
        self.theta = threshold

    def compute_single(self, *x: float) -> bool:
        arr = np.fromiter(x, dtype=float)
        res = self(arr)
        return res.item()
    
    def __call__(self, x: npt.NDArray[np.floating]) -> npt.NDArray[np.bool_]:
        if x.ndim == 1:
            x = x.reshape(1, -1)
        integrated = (self.weight*x).sum(axis=1)
        return integrated >= self.theta


1. Set $\theta = 1.5$ and $w_1 = w_2 = 1$. Fill in the truth table for all combinations of $x_1,x_2 \in \{0,1\}$. Which logical gate does this implement?

In [ ]:
perceptron = Perceptron(weight=[1, 1], threshold=1.5)

for input_ in itertools.product([True, False], repeat=2):
    print(*input_, perceptron.compute_single(*input_))

Behold, an AND-gate!
***

2. Change $\theta = 0.5$ with the same weights. Which logical gate does this now implement?

In [ ]:
perceptron2 = Perceptron(weight=[1, 1], threshold=0.5)

for input_ in itertools.product([True, False], repeat=2):
    print(*input_, perceptron2.compute_single(*input_))

BEHOLD, an OR-gate!
***

3. Can a single McCulloch–Pitts neuron implement the XOR function? Try to find weights and a threshold that work. If you cannot, explain geometrically why it is impossible.

Nah!

The integrated signal $\sum_i w_i x_i$ is linear, so it is either strictly increasing or decreasing with each $x_j$

To work as an XOR for e.g. $x = (0, 1)$, we need $w_1 > w_0$, but to work for $x = (1, 0)$, we need $w_0 > w_1$.

Geometrically, we can design any line through the grid with values in $[0, 1]$ for $x_0$ and $x_1$, such that values on one side give output True and the other side False. The inputs which give True for XOR are in the southwest and northeast corners of that grid, so such a line is not possible.
***


**Integrate and threshold**

A neuron receives two excitatory inputs with weights $w_1 = 0.6$ and $w_2 = 0.8$, and one inhibitory input with weight $w_3 = -1.0$. The firing threshold is $\theta = 0.5$.

1. Does the neuron fire when $x_1 = 1$, $x_2 = 0$, $x_3 = 1$?

In [ ]:
perceptron3 = Perceptron(weight=(0.6, 0.8, -1.0), threshold=0.5)
print(f"Will it fire?", perceptron3.compute_single(1, 0, 1))

2. Does the neuron fire when $x_1 = 1$, $x_2 = 1$, $x_3 = 0$?

In [ ]:
print(f"Will it fire... now?", perceptron3.compute_single(1, 1, 0))

3. The inhibitory input acts as a gate. Describe in words what role such gating plays epistemically — what does it mean for information processing that a neuron can be silenced by inhibition?

Hmmm not too sure about this one. Depending on the ration of the inhibitory weight to the remaining weights, it can mean that it's possible to ignore inputs below a certain threshold, or to shut it off completely (effectively using the inhibitory input to temporarily increase the threshold $\theta$).

Epistemically, I guess this is a way to encode context/attention? The degree to which the neuron's inputs propagate further becomes dependent on whatever causes the inhibitory input
***

**The four-step decision process and the Shannon channel**

Describe in your own words how each of the four steps of the neuron’s decision process (Receive, Integrate, Threshold, Propagate) maps onto the components of the Shannon–Weaver transmission model (source, transmitter, channel, receiver, destination). Is the mapping exact? Where does it break down?


* Receive -> **receiver**. Maps fairly nicely, as the receiver in the Shannon-Weaver Model (SWM) represents receiving the signal, separate from decoding it into a message. Also maps to **destination**, I guess, since neurons both send and receive signals.
* Integrate: Doesn't map thaaat well, I think? Mainly corresponds to the conversion from signal to message, going from _receiver_ to _destination_. Doesn't match that well because 1) it's somewhat analogous to not one component but a conversion in the SWM framework, and 2) whereas SWM is concerned with recovering or 'decoding' a message, integration purposely alters it (for example truncating signals when receiving an inhibitory input)
* Threshold: Doesn't map terribly well... Best match is probably the **transmitter**, as that is the component responsible for encoding a message into a signal. Doesn't map too well because thresholding is more of a decision mechanism (deciding whether to transmit or not).
* Propagate: Corresponds to sending a signal via a **channel**

***

### 1.2.2 Jupyter exercises

**Simulating the perceptron**

Implement the McCulloch–Pitts neuron in Python. Vary the threshold $\theta$ from $-2$ to $2$ in steps of $0.1$ and, for fixed inputs $(x_1,x_2) = (0.7,0.5)$ with $w_1 = w_2 = 1$, plot the output $y$ as a function of $\theta$. What kind of function does this trace out?

In [ ]:
def output_with_threshold(theta: float) -> float:
    return Perceptron(
        weight=np.array([1.0, 1.0]),
        threshold=theta
    ).compute_single(0.7, 0.5)


plotting.hacks.plot_function(output_with_threshold, xmin=-2, xmax=+2, n_points=41, ylabel="output");

It looks just like an inverted Heaviside step-function. That's because it is (it's computing $\Theta$(C - \theta), where $C$ is a constant from summin the weighted inputs, which are held fixed here)
***

### 1.2.3 Sigmoidal activation and the soft threshold

Replace the Heaviside function with a sigmoid:

$$
\sigma(z) = \frac{1}{1+e^{-\beta z}},
\qquad
z = \sum_i w_i x_i - \theta
\tag{3}
$$

Plot $\sigma(z)$ for $\beta = 0.5, 1, 5, 20$, all in the same coordinate system. What happens as $\beta \to \infty$? What does this suggest about the relationship between the soft and hard threshold neuron models?

In [ ]:
_, ax = plt.subplots()
zvals = np.linspace(-2, 2, num=100)
for beta in (0.5, 1, 5, 20):
    ax.plot(
        zvals,
        1.0/(1.0 + np.exp(-beta*zvals)),
        label=f"$\\beta = {beta}$"
    )

ax.legend();

The sigmoid approaches the Heaviside, so the output approaches a step function
***

## 1.3 Graphs and Neural Connectivity

### 1.3.1 Pen-and-paper exercises

**Adjacency matrices**

Consider a directed graph with four nodes $\{A,B,C,D\}$ and edges: $A \to B$, $A \to C$, $B \to D$, $C \to B$, $D \to A$.


1. Write down the $4 \times 4$ adjacency matrix $A$, where $A_{ij} = 1$ if there is a directed edge from node $j$ to node $i$.

In [ ]:
nodes = ("A", "B", "C", "D")
links = (
    ("A", "B"),
    ("A", "C"),
    ("B", "D"),
    ("C", "B"),
    ("D", "A"),
)
node_to_index = {node: i for i, node in enumerate(nodes)}
A = np.zeros(shape=(4, 4), dtype=int)
for link in links:
    u, v = map(node_to_index.get, link)
    A[u, v] = 1

In [ ]:
print(A)

***

2. Compute $A^2$. What does the $(i,j)$ entry of $A^2$ represent in terms of paths in the network?

3. Which nodes can be reached from node A in exactly two steps?

In [ ]:
A_squared = A @ A
print(A_squared)

reachable_inds, = np.where(A_squared[node_to_index["A"]] != 0)
print("Reachable nodes:", ', '.join((nodes[ind] for ind in reachable_inds)))

The $(i, j)$ entry represents whether node $j$ can be reached from node $i$ in 2 steps (or $n$ steps for $A^n$).
Nodes B and D are reachable from A in 2 steps.
***

**Signed graphs and excitation/inhibition**

Extend the graph above by assigning weights: the edge $C \to B$ is inhibitory (weight $-1$) and all other edges are excitatory (weight $+1$).


1. Write the signed adjacency matrix.

In [ ]:
A[node_to_index["C"], node_to_index["B"]] = -1
print(A)

2. Consider a signal originating at A with strength $+1$. Trace its sign as it propagates along the path $A \to C \to B \to D$. Is the net effect at D excitatory or inhibitory?

In [ ]:
# NOTE: Bit unclear, but guessing we have to multiply the weights together along the path?

path = ("A", "C", "B", "D")
signal = +1
for link in itertools.pairwise(path):
    i, j = map(node_to_index.__getitem__, link)
    weight_ = A[i, j]
    signal *= weight_
    print(f"Link {' -> '.join(link)} (weight={weight_}). Signal is now {signal}")


Net effect at D is inhibitory (more A -> more C -> less B -> less D)
***

3. What is the biological interpretation of a negative entry in the adjacency matrix?

The negative entry for the $C \rightarrow B$ edge means that neuron $C$ releases an inhibitory neurotransmitter like GABA from its active zone into the synaptic cleft between itself and neuron $B$
***

**Neuron doctrine vs. ensemble theory**

A classical debate contrasts the Neuron Doctrine with Ensemble Theory.

1. In graph terms, how would a “grandmother cell” (one neuron encoding one concept) differ from a distributed ensemble code? Sketch both as small graphs with labelled nodes.

**Neuron Doctrine:** The 'Grandmother cell' is a single node which represents some abstract concept.
Fires or not depending on whether its input matches the concept corresponding to the node/neuron. Outputs from the node are all-or-nothing.

**Ensemble Theory:** A concept is represented by an ensemble of neurons/nodes, each representing various aspects of the concept. I'm guessing the outputs of the ensemble can be graduated (if only a subsets of neurons fire).
***

2. What is the consequence for robustness when a single node is removed from each representation?

ND: The ability to encode the concept is lost.

ET: The encoding of some aspects is lost, but will likely not change a lot (only in cases where some neuron receives its inputs from the removed one and is close to its threshold)

3. Briefly relate this to Shannon’s concept of redundancy. Which coding scheme is more redundant?

Redundancy: Encoding more information than absolutely necessary, so the message can be reconstructed even with some degree of information loss due to noise. The ensemble is more redundant here
***

### 1.3.2 Jupyter exercises

**Constructing and visualising graphs**

Using networkx, construct the directed graph from the pen-and-paper exercise. Visualise it with labelled nodes. Compute and print the in-degree and out-degree of each node. Which node has the highest in-degree?

In [ ]:
G: nx.DiGraph[str] = nx.DiGraph()
G.add_nodes_from(nodes)
G.add_edges_from(links)

_, ax = plt.subplots()

nx.draw(
    G,
    ax=ax,
    with_labels=True,
    horizontalalignment="center",
    verticalalignment="center",
    font_size=10,
    node_color=[plotting.pyplot_utils.cycle_color(ax)]*len(G),
    pos=nx.circular_layout(G)
)

In [ ]:
print("\n".join((f"{node_}: {deg_}" for node_, deg_ in G.in_degree())))

Node $B$ has the highest in-degree
***



**Random networks and degree distributions**

Generate an Erdős–Rényi random graph with $n = 100$ nodes and connection probability $p = 0.05$. Plot the degree distribution as a histogram. Repeat for $p = 0.2$. How does the shape change? Now compare with a scale-free network (`networkx.barabasi_albert_graph`). Plot the degree distribution on a double-logarithmic scale. What do you observe, and what might this mean biologically?

## 1.4 The Hopfield Network and Associative Memory

### 1.4.1 Pen-and-paper exercises

**Energy function**

The Hopfield network energy is defined as

$$
E = -\frac{1}{2}\sum_{i \ne j} w_{ij}s_i s_j
\tag{4}
$$

where $s_i \in \{-1,+1\}$.

1. Consider a two-neuron network with $w_{12} = w_{21} = 1$. Compute $E$ for all four states: $(+1,+1)$, $(+1,-1)$, $(-1,+1)$, $(-1,-1)$. Which states are energy minima?

2. Repeat for $w_{12} = w_{21} = -1$. Which states are now the minima? What does a negative weight mean biologically?

3. Explain in words why energy minima correspond to stored memories and why the network retrieves them even from incomplete input.

**Hebbian learning and weight matrices**

Memories are stored via the Hebbian rule:

$$
w_{ij} = \frac{1}{N}\sum_{\mu=1}^{P}\xi_i^\mu \xi_j^\mu,
\qquad w_{ii} = 0
\tag{5}
$$

where $\xi^\mu \in \{-1,+1\}^N$ is the $\mu$-th stored pattern and $P$ is the number of patterns.

1. Store two patterns in a $N=4$ neuron network: $\xi^1 = (+1,+1,-1,-1)$ and $\xi^2 = (+1,-1,+1,-1)$. Compute the full weight matrix $W$.

2. Starting from the corrupted probe $s = (+1,+1,+1,-1)$ (one bit flipped from $\xi^1$), perform one step of synchronous update using

$$
s_i \leftarrow \operatorname{sign}\left(\sum_j w_{ij}s_j\right)
\tag{6}
$$

Does the network recover $\xi^1$?

3. Roughly how many patterns can a Hopfield network of $N=100$ neurons reliably store? State the theoretical estimate.

### 1.4.2 Jupyter exercises

**Simulating a Hopfield network**

Implement a Hopfield network in Python. Store three random binary patterns of length $N=100$. Starting from a probe with 20% of bits flipped, run synchronous updates until convergence. Plot the overlap

$$
m^\mu(t) = \frac{1}{N}\sum_i \xi_i^\mu s_i(t)
$$

as a function of update step for each stored pattern. Does the network converge to the correct memory?

**Storage capacity**

The theoretical storage capacity is approximately

$$
P_{\max} \approx 0.138N.
$$

Verify this empirically: for $N=200$, vary the number of stored patterns $P$ from 1 to 50. For each value of $P$, store $P$ random patterns, present a noisy probe (10% bit flips), and record whether retrieval is successful (overlap $>0.95$). Plot the retrieval success rate as a function of $P/N$. Does the observed transition match the theoretical prediction?

## 1.5 Emergence and Integrated Information

### 1.5.1 Pen-and-paper exercises

**Decomposability and $\Phi$**

Integrated Information Theory defines $\Phi$ as the information generated by a system above and beyond its parts. A system with $\Phi = 0$ is fully decomposable.

1. Consider two neurons A and B with no connections between them. Is $\Phi = 0$ or $\Phi > 0$? Justify your answer.

2. Now connect them bidirectionally with equal weights. Qualitatively, how does this affect $\Phi$?

3. Heidegger argued that a car cannot be understood by decomposing it into parts. Explain how this intuition maps onto the concept of $\Phi$. Where does the analogy hold and where does it fail?

**Information processing across scales**

Information in neural systems is processed simultaneously at multiple spatial and temporal scales.

1. Give one concrete example of a computation best described at the molecular scale and one best described at the network scale. Why is neither description alone sufficient?

2. The PKA/ERK signalling cascade can be modelled as a dynamical system. What are the “inputs” and “outputs” at this scale, and how do they relate to inputs and outputs at the single-neuron scale?

3. In your own words, what is the central challenge of property integration across scales in theoretical neuroscience?

### 1.5.2 Jupyter exercises

**Simulating emergence in a ring network**

Consider a ring of $N=20$ binary neurons, each connected to its two neighbours with weight $w=1$ and threshold $\theta=1.5$ (a neuron fires only if both neighbours are active). Initialise the ring with a random pattern.

1. Simulate synchronous updates for 50 steps and display the activity of all neurons over time as a raster plot (neurons on the y-axis, time steps on the x-axis).

2. Try several different random initial conditions. Do you observe any recurring patterns or attractors?

3. Now disconnect every other neuron (set their weights to zero), splitting the ring into two independent halves. Compare the dynamics to the intact ring. Does the emergent behaviour re-appear in either half?

# 2 Single neuron models

## 2.1 Nernst potentials

Calculate the reversal potentials (Nernst potentials) of the following electrolytes, given the concentrations in the table below. Fill out your results in the second table, including the direction of movement when the potential is at the equilibrium membrane potential, assuming $-70\,\mathrm{mV}$.

| Electrolyte | $C_{\mathrm{in}}$ (mM) | $C_{\mathrm{out}}$ (mM) |
|:---:|---:|---:|
| Na$^+$ | 15 | 145 |
| K$^+$ | 140 | 5 |
| Cl$^+$ | 10 | 110 |
| Ca$^{2+}$ | 0.0001 | 2 |

Fill out the table below:

| Electrolyte | Nernst potential (mV) | Direction of movement at $-70\,\mathrm{mV}$ |
|:---:|:---:|:---:|
| Na$^+$ | | |
| K$^+$ | | |
| Cl$^+$ | | |
| Ca$^{2+}$ | | |

What is the resulting equilibrium potential, $V_m$, for the neuron? Hint: use Millman’s equation.

$$
V_m = ?
$$

## 2.2 Electronic circuit components

- Discuss what the resistor/resistance, $R$, represents in the electronic model of the neuronal membrane.

- What does the conductance, $G$, represent?

### 2.2.1 Membrane equation

Write up the membrane equation. What are the different contributions? Discuss with fellow student(s). How does synaptic input come into the equation? Does it have a reversal potential? And what is the reversal potential of synaptic input?

### 2.2.2 Finding the time constant, $\tau_m$

If we look at the membrane potential excursion from steady-state membrane potential:

$$
V_m = V_0 e^{t/\tau_m}
\tag{7}
$$

If, for instance, a synaptic input moves $V_m$ away from steady state, the potential will decay back to steady-state potential with a time constant $\tau_m$. Three neurons, which have different time constants, are examined in the figure below.

Find $\tau_m$ for each of them by visual inspection.

**Fig. 1.** Measuring decay of membrane potential of three different neurons. What are their time constants?